In [1]:
# Install dependencies
# python -m pip install -r requirements.txt

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from pyproj import Transformer

import pulp

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [3]:
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / "data" / "processed"
assert DATA_DIR.is_dir(), (
    f"{DATA_DIR.resolve()} not found -- run this notebook from the "
    f"notebooks/ folder of the alaska-fuel-network repo checkout."
)

REFINERY_FILE = DATA_DIR / "refineries.csv"
IMPORT_HUB_FILE = DATA_DIR / "marine_import_hubs.csv"
DIST_FILE = DATA_DIR / "distribution_terminals.csv"
LAST_MILE_FILE = DATA_DIR / "last_mile_locations.csv"

for file_path in [
    REFINERY_FILE,
    IMPORT_HUB_FILE,
    DIST_FILE,
    LAST_MILE_FILE,
]:
    print(file_path.name, "exists:", file_path.exists())

refineries.csv exists: True
marine_import_hubs.csv exists: True
distribution_terminals.csv exists: True
last_mile_locations.csv exists: True


In [4]:
refineries_raw = pd.read_csv(REFINERY_FILE)
import_hubs_raw = pd.read_csv(IMPORT_HUB_FILE)
dist_raw = pd.read_csv(DIST_FILE)
last_mile_raw = pd.read_csv(LAST_MILE_FILE)

print("Refinery rows:", len(refineries_raw))
print("Marine import hub rows:", len(import_hubs_raw))
print("Distribution rows:", len(dist_raw))
print("Last-mile rows:", len(last_mile_raw))

Refinery rows: 5
Marine import hub rows: 11
Distribution rows: 73
Last-mile rows: 931


In [5]:
refineries = (
    refineries_raw[
        [
            "refinery_id",
            "facility_name",
            "latitude",
            "longitude",
            "model_capacity",
        ]
    ]
    .rename(
        columns={
            "refinery_id": "node_id",
            "facility_name": "node_name",
        }
    )
    .copy()
)

refineries["node_type"] = "R"


import_hubs = (
    import_hubs_raw[
        [
            "hub_id",
            "hub_name",
            "latitude",
            "longitude",
            "model_capacity",
        ]
    ]
    .rename(
        columns={
            "hub_id": "node_id",
            "hub_name": "node_name",
        }
    )
    .copy()
)

import_hubs["node_type"] = "P"


suppliers = pd.concat(
    [refineries, import_hubs],
    ignore_index=True,
)

suppliers.head()

,node_id,node_name,latitude,longitude,model_capacity,node_type
0,R_KENAI,Kenai Refinery,60.6842,-151.3672,"123,913.7174",R
1,R_PETROSTAR_NORTH_POLE,Petro Star North Pole Refinery,64.7327,-147.3453,"45,556.5138",R
2,R_PETROSTAR_VALDEZ,Petro Star Valdez Refinery,61.0847,-146.2531,"113,891.2844",R
3,R_CONOCOPHILLIPS_KUPARUK,ConocoPhillips Kuparuk/Prudhoe Bay topping pla...,70.3245,-149.5990,"27,333.9083",R
4,R_HILCORP_PRUDHOE,Hilcorp North Slope Crude Oil Topping Unit,70.2550,-148.3472,"11,844.6936",R


In [6]:
distribution = (
    dist_raw[
        [
            "TERM_ID",
            "NAME",
            "LATITUDE",
            "LONGITUDE",
            "model_capacity",
        ]
    ]
    .rename(
        columns={
            "TERM_ID": "node_id",
            "NAME": "node_name",
            "LATITUDE": "latitude",
            "LONGITUDE": "longitude",
        }
    )
    .copy()
)

distribution["node_type"] = "D"

# Rows excluded from the terminal-capacity calculation should already
# have model_capacity equal to zero.
distribution = distribution[
    distribution["model_capacity"] > 0
].reset_index(drop=True)

distribution.head()

,node_id,node_name,latitude,longitude,model_capacity,node_type
0,ANLTK02001,ALEUT ENTERPRISE CORPORATION (ADAK ISLAND),51.8708,-176.6422,"2,539.2857",D
1,ANLTK02002,ANCHORAGE FUELING AND SERVICE COMPANY OFF-AIRP...,61.2343,-149.8850,"37,816.9126",D
2,ANLTK02003,ANCHORAGE FUELING AND SERVICE COMPANY AIRPORT ...,61.1749,-150.0113,"61,774.7861",D
3,ANLTK02004,CROWLEY FUELS LLC - ANCHORAGE BULK FUEL FACILITY,61.2312,-149.8882,"75,863.2914",D
4,ANLTK02005,TESORO LOGISTICS OPERATIONS LLC OCEAN DOCK PET...,61.2302,-149.8937,"48,073.9998",D


In [7]:
customers = (
    last_mile_raw[
        [
            "cluster_id",
            "Name",
            "LATITUDE",
            "LONGITUDE",
            "model_capacity",
        ]
    ]
    .rename(
        columns={
            "cluster_id": "node_id",
            "Name": "node_name",
            "LATITUDE": "latitude",
            "LONGITUDE": "longitude",
            "model_capacity": "demand",
        }
    )
    .copy()
)

customers["node_type"] = "L"

customers = customers[
    customers["demand"] > 0
].reset_index(drop=True)

customers.head()

,node_id,node_name,latitude,longitude,demand,node_type
0,L_0001,"""Alice Mae's"" Shoppers Cache - The Laundry",61.3941,-149.4634,238.0952,L
1,L_0002,10th Street Tesoro,58.3018,-133.5764,571.4286,L
2,L_0003,10th Street Tesoro | Tesoro Alaska,58.3016,-134.4238,571.4286,L
3,L_0004,2 Go Mart-Tesoro Northstore #0077,61.5938,-149.1211,380.9524,L
4,L_0005,2 Go Mart-Tesoro Northstore #0079,61.1948,-149.8675,619.0476,L


In [8]:
dist_work = dist_raw.copy()

numeric_columns = [
    "CAPACITY",
    "refined_capacity_weight_barrels",
    "model_capacity",
    "LATITUDE",
    "LONGITUDE",
]

for column in numeric_columns:
    dist_work[column] = pd.to_numeric(
        dist_work[column],
        errors="coerce",
    )

In [9]:
status_is_operating = (
    dist_work["STATUS"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("IN SERVICE")
)

 #Refined-product service is established from the record's own evidence.
_refined_product_flags = ["GASOLINE", "DISTILLATE", "JETFUEL", "AVGAS"]

handles_refined_product = (
    dist_work["REFINED"].astype(str).str.strip().str.upper().eq("YES")
    | dist_work["COMMODITY"].astype(str).str.upper().str.contains(
        "REFINED", na=False
    )
    | pd.concat(
        [
            dist_work[flag].astype(str).str.strip().str.upper().eq("YES")
            for flag in _refined_product_flags
        ],
        axis=1,
    ).any(axis=1)
)

is_crude_system_type = (
    dist_work["TYPE"]
    .astype(str)
    .str.upper()
    .str.contains("CRUDE", na=False)
)

has_positive_storage = (
    dist_work["refined_capacity_weight_barrels"] > 0
)

has_coordinates = (
    dist_work["LATITUDE"].notna()
    & dist_work["LONGITUDE"].notna()
)

In [10]:
excluded_crude_system = dist_work[
    status_is_operating
    & handles_refined_product
    & is_crude_system_type
    & has_positive_storage
].copy()

excluded_crude_system[
    [
        "TERM_ID",
        "NAME",
        "CITY",
        "TYPE",
        "COMMODITY",
        "CAPACITY",
        "refined_capacity_weight_barrels",
        "model_capacity",
    ]
]

,TERM_ID,NAME,CITY,TYPE,COMMODITY,CAPACITY,refined_capacity_weight_barrels,model_capacity


In [11]:
eligible_distribution_mask = (
    status_is_operating
    & handles_refined_product
    & ~is_crude_system_type
    & has_positive_storage
    & has_coordinates
)

distribution = (
    dist_work.loc[
        eligible_distribution_mask,
        [
            "TERM_ID",
            "NAME",
            "CITY",
            "TYPE",
            "STATUS",
            "REFINED",
            "CRUDE_OIL",
            "CAPACITY",
            "refined_capacity_weight_barrels",
            "LATITUDE",
            "LONGITUDE",
            "model_capacity",
        ],
    ]
    .rename(
        columns={
            "TERM_ID": "node_id",
            "NAME": "node_name",
            "CAPACITY": "reported_storage_capacity_barrels",
            "refined_capacity_weight_barrels": "storage_capacity_barrels",
            "LATITUDE": "latitude",
            "LONGITUDE": "longitude",
            "model_capacity": "previous_model_capacity",
        }
    )
    .copy()
)

distribution["node_type"] = "D"

distribution = distribution.reset_index(drop=True)

In [12]:
# Distribution-layer capacity buffer.
RHO_D = 0.50

# Regional service-area limits prevent implausible long-distance assignments
# from western marine hubs and Yukon River terminals. Red Dog is excluded as a
# specialized mine port. See config/domain_rules.json and the evidence audit.
SERVICE_RADIUS_KM = {
    "ANLTK02035": 300.0,   # Crowley Marine Services, Kotzebue
    "ANLTK02048": 250.0,   # Crowley Marine Services, Nome
    "ANLTK02045": 250.0,   # Bonanza Fuel, Nome
    "ANLTK02047": 250.0,   # Crowley Marine Services, Nome (second berth)
    "ANLTK02021": 250.0,   # Crowley Fuels, Fort Yukon (Yukon River)
    "ANLTK02022": 250.0,   # Crowley Marine Services, Galena (Yukon River)
}

KUPARUK_L = "L_KUPARUK_INDUSTRIAL_OPERATIONS"
PRUDHOE_L = "L_PRUDHOE_DEADHORSE_INDUSTRIAL_OPERATIONS"
ADAK_L = "L_0039"
ST_PAUL_L = "L_ST_PAUL_COMMUNITY_FUEL"
ST_GEORGE_L = "L_ST_GEORGE_COMMUNITY_FUEL"

KUPARUK_D = "D_KUPARUK_LOCAL"
PRUDHOE_D = "ANLTK02054"
ADAK_D = "ANLTK02001"
ST_PAUL_D = "ANLTK02063"
ST_GEORGE_D = "ANLTK02062"

customer_demand_by_id = customers.set_index("node_id")["demand"]
required_local_customers = {
    KUPARUK_L, PRUDHOE_L, ADAK_L, ST_PAUL_L, ST_GEORGE_L
}
missing_local_customers = (
    required_local_customers - set(customer_demand_by_id.index)
)
assert not missing_local_customers, (
    f"Missing local L nodes: {sorted(missing_local_customers)}"
)

kuparuk_demand = float(customer_demand_by_id[KUPARUK_L])
prudhoe_demand = float(customer_demand_by_id[PRUDHOE_L])
adak_demand = float(customer_demand_by_id[ADAK_L])
st_paul_demand = float(customer_demand_by_id[ST_PAUL_L])
st_george_demand = float(customer_demand_by_id[ST_GEORGE_L])
total_demand = float(customers["demand"].sum())
target_total_distribution_capacity = total_demand * (1 + RHO_D)

fixed_physical_demand = {
    PRUDHOE_D: prudhoe_demand,
    ADAK_D: adak_demand,
    ST_PAUL_D: st_paul_demand,
    ST_GEORGE_D: st_george_demand,
}
fixed_physical_ids = set(fixed_physical_demand)
fixed_physical_terminals = distribution.loc[
    distribution["node_id"].isin(fixed_physical_ids)
].copy()
assert set(fixed_physical_terminals["node_id"]) == fixed_physical_ids

fixed_physical_terminals.loc[:, "model_capacity"] = (
    fixed_physical_terminals["node_id"].map(fixed_physical_demand)
    * (1 + RHO_D)
)
fixed_physical_terminals.loc[:, "capacity_weight"] = 0.0

prudhoe_terminal = fixed_physical_terminals.loc[
    fixed_physical_terminals["node_id"].eq(PRUDHOE_D)
].copy()
kuparuk_terminal = prudhoe_terminal.copy()
kuparuk_terminal.loc[:, "node_id"] = KUPARUK_D
kuparuk_terminal.loc[:, "node_name"] = (
    "Kuparuk Local Refined-Fuel Terminal"
)
kuparuk_terminal.loc[:, "CITY"] = "Kuparuk"
kuparuk_terminal.loc[:, "TYPE"] = (
    "Representative Local Refinery Terminal"
)
kuparuk_terminal.loc[:, "STATUS"] = "IN SERVICE"
kuparuk_terminal.loc[:, "REFINED"] = "YES"
kuparuk_terminal.loc[:, "CRUDE_OIL"] = "NO"
kuparuk_terminal.loc[:, "storage_capacity_barrels"] = np.nan
kuparuk_terminal.loc[:, "latitude"] = 70.3245
kuparuk_terminal.loc[:, "longitude"] = -149.5990
kuparuk_terminal.loc[:, "previous_model_capacity"] = np.nan
kuparuk_terminal.loc[:, "model_capacity"] = (
    (1 + RHO_D) * kuparuk_demand
)
kuparuk_terminal.loc[:, "capacity_weight"] = 0.0

# --- terminals with a hard service radius -------------------------------
def _haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0088
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    d_phi = phi2 - phi1
    d_lambda = np.radians(lon2 - lon1)
    haversine = (
        np.sin(d_phi / 2) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(d_lambda / 2) ** 2
    )
    return 2 * earth_radius_km * np.arcsin(np.sqrt(haversine))


radius_ids = set(SERVICE_RADIUS_KM) & set(distribution["node_id"])
missing_radius_ids = set(SERVICE_RADIUS_KM) - radius_ids
assert not missing_radius_ids, (
    f"SERVICE_RADIUS_KM references terminals absent from the eligible D set: "
    f"{sorted(missing_radius_ids)}"
)
assert not (radius_ids & fixed_physical_ids), (
    "a terminal cannot be both radius-limited and pinned to a local customer"
)

radius_reachable_demand = {}
for terminal_id in sorted(radius_ids):
    terminal_row = distribution.loc[
        distribution["node_id"].eq(terminal_id)
    ].iloc[0]
    distance_km = _haversine_km(
        terminal_row["latitude"],
        terminal_row["longitude"],
        customers["latitude"].to_numpy(),
        customers["longitude"].to_numpy(),
    )
    within = distance_km <= SERVICE_RADIUS_KM[terminal_id]
    radius_reachable_demand[terminal_id] = float(
        customers.loc[within, "demand"].sum()
    )
    assert radius_reachable_demand[terminal_id] > 0, (
        f"{terminal_id} can reach no demand within "
        f"{SERVICE_RADIUS_KM[terminal_id]} km"
    )

# A radius-limited terminal keeps its storage-based share unless that share
# exceeds the demand it is allowed to reach, in which case it is capped at
# (1 + RHO_D) x reachable demand and the excess is redistributed.
# village terminal ~95,000 barrels of model capacity.
other_distribution = distribution.loc[
    ~distribution["node_id"].isin(fixed_physical_ids)
].copy()
fixed_local_capacity = float(
    fixed_physical_terminals["model_capacity"].sum()
    + kuparuk_terminal["model_capacity"].sum()
)
remaining_distribution_capacity = (
    target_total_distribution_capacity - fixed_local_capacity
)
assert remaining_distribution_capacity > 0

other_storage_total = float(
    other_distribution["storage_capacity_barrels"].sum()
)
assert other_storage_total > 0
other_distribution.loc[:, "capacity_weight"] = (
    other_distribution["storage_capacity_barrels"]
    / other_storage_total
)
other_distribution.loc[:, "model_capacity"] = (
    remaining_distribution_capacity
    * other_distribution["capacity_weight"]
)

radius_ceiling = {
    terminal_id: (1 + RHO_D) * reachable
    for terminal_id, reachable in radius_reachable_demand.items()
}
is_capped = other_distribution["node_id"].map(radius_ceiling).notna() & (
    other_distribution["node_id"].map(radius_ceiling)
    < other_distribution["model_capacity"]
)
freed_capacity = float(
    (
        other_distribution.loc[is_capped, "model_capacity"]
        - other_distribution.loc[is_capped, "node_id"].map(radius_ceiling)
    ).sum()
)
other_distribution.loc[is_capped, "model_capacity"] = (
    other_distribution.loc[is_capped, "node_id"].map(radius_ceiling)
)
other_distribution.loc[is_capped, "capacity_weight"] = 0.0

if freed_capacity > 0:
    grows = ~is_capped
    growth_weight = (
        other_distribution.loc[grows, "model_capacity"]
        / other_distribution.loc[grows, "model_capacity"].sum()
    )
    other_distribution.loc[grows, "model_capacity"] += (
        freed_capacity * growth_weight
    )

print(f"radius ceiling binds on {int(is_capped.sum())} terminal(s); "
      f"{freed_capacity:,.1f} bbl redistributed")

distribution = pd.concat(
    [other_distribution, fixed_physical_terminals, kuparuk_terminal],
    ignore_index=True,
)

print(f"RHO_D = {RHO_D}: target D capacity "
      f"{target_total_distribution_capacity:,.2f} for demand {total_demand:,.2f}")
for terminal_id in sorted(radius_ids):
    print(f"  radius-limited {terminal_id}: "
          f"{SERVICE_RADIUS_KM[terminal_id]:.0f} km, reachable demand "
          f"{radius_reachable_demand[terminal_id]:,.1f}, capacity "
          f"{float(distribution.loc[distribution['node_id'].eq(terminal_id), 'model_capacity'].iloc[0]):,.1f}")

radius ceiling binds on 5 terminal(s); 20,140.5 bbl redistributed
RHO_D = 0.5: target D capacity 587,518.75 for demand 391,679.16
  radius-limited ANLTK02021: 250 km, reachable demand 63,315.0, capacity 1,158.2
  radius-limited ANLTK02022: 250 km, reachable demand 900.5, capacity 1,350.7
  radius-limited ANLTK02035: 300 km, reachable demand 2,488.1, capacity 3,732.1
  radius-limited ANLTK02045: 250 km, reachable demand 3,420.8, capacity 5,131.3
  radius-limited ANLTK02047: 250 km, reachable demand 3,420.8, capacity 5,131.3
  radius-limited ANLTK02048: 250 km, reachable demand 3,420.8, capacity 5,131.3


In [13]:
print("Eligible refined-product D nodes:", len(distribution))

print(
    "Target D capacity:",
    f"{target_total_distribution_capacity:,.4f}",
)

print(
    "Calculated D capacity:",
    f"{distribution['model_capacity'].sum():,.4f}",
)

print(
    "Demand:",
    f"{total_demand:,.4f}",
)

assert distribution["node_id"].is_unique

assert np.isclose(
    distribution["model_capacity"].sum(),
    target_total_distribution_capacity,
    rtol=1e-10,
    atol=1e-4,
)

Eligible refined-product D nodes: 57
Target D capacity: 587,518.7466
Calculated D capacity: 587,518.7466
Demand: 391,679.1644


In [14]:
def validate_nodes(df, capacity_column, table_name):
    required = [
        "node_id",
        "latitude",
        "longitude",
        capacity_column,
    ]

    print(f"\n{table_name}")
    print("-" * len(table_name))

    print("Number of nodes:", len(df))
    print("Duplicate node IDs:", df["node_id"].duplicated().sum())
    print("Missing node IDs:", df["node_id"].isna().sum())
    print("Missing latitudes:", df["latitude"].isna().sum())
    print("Missing longitudes:", df["longitude"].isna().sum())
    print(
        f"Missing {capacity_column}:",
        df[capacity_column].isna().sum(),
    )
    print(
        f"Nonpositive {capacity_column}:",
        (df[capacity_column] <= 0).sum(),
    )

    assert set(required).issubset(df.columns)
    assert not df["node_id"].duplicated().any()
    assert df[required].notna().all().all()


validate_nodes(
    suppliers,
    capacity_column="model_capacity",
    table_name="Suppliers",
)

validate_nodes(
    distribution,
    capacity_column="model_capacity",
    table_name="Distribution terminals",
)

validate_nodes(
    customers,
    capacity_column="demand",
    table_name="Last-mile customers",
)


Suppliers
---------
Number of nodes: 16
Duplicate node IDs: 0
Missing node IDs: 0
Missing latitudes: 0
Missing longitudes: 0
Missing model_capacity: 0
Nonpositive model_capacity: 0

Distribution terminals
----------------------
Number of nodes: 57
Duplicate node IDs: 0
Missing node IDs: 0
Missing latitudes: 0
Missing longitudes: 0
Missing model_capacity: 0
Nonpositive model_capacity: 0

Last-mile customers
-------------------
Number of nodes: 931
Duplicate node IDs: 0
Missing node IDs: 0
Missing latitudes: 0
Missing longitudes: 0
Missing demand: 0
Nonpositive demand: 0


In [15]:
total_supply = suppliers["model_capacity"].sum()
total_demand = customers["demand"].sum()
total_distribution_capacity = distribution["model_capacity"].sum()

dummy_demand = total_supply - total_demand

print(f"Total supplier capacity:     {total_supply:,.4f} barrels")
print(f"Total customer demand:       {total_demand:,.4f} barrels")
print(
    f"Total distribution capacity: "
    f"{total_distribution_capacity:,.4f} barrels"
)
print(f"Required dummy demand:       {dummy_demand:,.4f} barrels")

assert total_supply >= total_demand, (
    "Total supply is below total demand. "
    "This would require a dummy supplier instead of a dummy sink."
)

assert total_distribution_capacity >= total_demand, (
    "Distribution-terminal capacity is insufficient "
    "to carry the required customer flow."
)

Total supplier capacity:     430,847.0809 barrels
Total customer demand:       391,679.1644 barrels
Total distribution capacity: 587,518.7466 barrels
Required dummy demand:       39,167.9164 barrels


In [16]:
supply_summary = (
    suppliers.groupby("node_type", as_index=False)
    .agg(
        number_of_nodes=("node_id", "count"),
        total_model_capacity=("model_capacity", "sum"),
    )
)

supply_summary["share"] = (
    supply_summary["total_model_capacity"] / total_supply
)

supply_summary

,node_type,number_of_nodes,total_model_capacity,share
0,P,11,"108,306.9634",0.2514
1,R,5,"322,540.1174",0.7486


In [17]:
from pyproj import Geod

geod = Geod(ellps="WGS84")

In [18]:
def pairwise_geodesic_distance(origins, destinations):
    """
    Calculate all origin-destination geodesic distances.

    Parameters
    ----------
    origins : pandas.DataFrame
        Must contain node_id, latitude, and longitude.

    destinations : pandas.DataFrame
        Must contain node_id, latitude, and longitude.

    Returns
    -------
    pandas.DataFrame
        Pairwise WGS84 ellipsoidal distances in kilometers.
        Rows represent origins and columns represent destinations.
    """

    n_origins = len(origins)
    n_destinations = len(destinations)

    # Create one coordinate pair for every possible arc
    origin_lons = np.repeat(
        origins["longitude"].to_numpy(dtype=float),
        n_destinations,
    )

    origin_lats = np.repeat(
        origins["latitude"].to_numpy(dtype=float),
        n_destinations,
    )

    destination_lons = np.tile(
        destinations["longitude"].to_numpy(dtype=float),
        n_origins,
    )

    destination_lats = np.tile(
        destinations["latitude"].to_numpy(dtype=float),
        n_origins,
    )

    # Geod.inv returns:
    # forward azimuth, reverse azimuth, distance in meters
    _, _, distance_meters = geod.inv(
        origin_lons,
        origin_lats,
        destination_lons,
        destination_lats,
    )

    distance_km = np.asarray(distance_meters).reshape(
        n_origins,
        n_destinations,
    ) / 1_000

    return pd.DataFrame(
        distance_km,
        index=origins["node_id"].astype(str),
        columns=destinations["node_id"].astype(str),
    )

In [19]:
distance_supplier_to_dist = pairwise_geodesic_distance(
    origins=suppliers,
    destinations=distribution,
)

print(
    "Supplier-to-distribution matrix shape:",
    distance_supplier_to_dist.shape,
)

distance_supplier_to_dist.iloc[:5, :5]

Supplier-to-distribution matrix shape: (16, 57)


node_id,ANLTK02002,ANLTK02003,ANLTK02004,ANLTK02005,ANLTK02006
node_id,,,,,
R_KENAI,101.0205,91.6335,100.6748,100.3788,100.7869
R_PETROSTAR_NORTH_POLE,410.5627,418.9031,410.9462,411.1308,410.6822
R_PETROSTAR_VALDEZ,196.2134,202.7384,196.3652,196.6514,196.4680
R_CONOCOPHILLIPS_KUPARUK,"1,013.6639","1,020.3643","1,014.0141","1,014.1202","1,013.7121"
R_HILCORP_PRUDHOE,"1,008.2156","1,015.2234","1,008.5733","1,008.6930","1,008.2759"


In [20]:
distance_dist_to_customer = pairwise_geodesic_distance(
    origins=distribution,
    destinations=customers,
)

print(
    "Distribution-to-customer matrix shape:",
    distance_dist_to_customer.shape,
)

distance_dist_to_customer.iloc[:5, :5]

Distribution-to-customer matrix shape: (57, 931)


node_id,L_0001,L_0002,L_0003,L_0004,L_0005
node_id,,,,,
ANLTK02002,28.7654,969.7772,925.4550,57.1733,4.4953
ANLTK02003,38.2056,975.0204,930.5110,66.6490,8.0431
ANLTK02004,29.1169,969.8698,925.5387,57.5406,4.1975
ANLTK02005,29.4094,970.1332,925.7981,57.8204,4.1870
ANLTK02006,28.9957,970.0189,925.6943,57.3882,4.5123


In [21]:
def validate_distance_matrix(
    distance_matrix,
    expected_origins,
    expected_destinations,
    matrix_name,
):
    values = distance_matrix.to_numpy()

    print(f"\n{matrix_name}")
    print("-" * len(matrix_name))
    print("Shape:", distance_matrix.shape)
    print("Minimum distance:", f"{values.min():,.4f} km")
    print("Maximum distance:", f"{values.max():,.4f} km")
    print("Mean distance:", f"{values.mean():,.4f} km")
    print("Missing values:", np.isnan(values).sum())
    print("Negative values:", (values < 0).sum())

    assert distance_matrix.shape == (
        expected_origins,
        expected_destinations,
    )
    assert np.isfinite(values).all()
    assert (values >= 0).all()


validate_distance_matrix(
    distance_supplier_to_dist,
    expected_origins=len(suppliers),
    expected_destinations=len(distribution),
    matrix_name="Supplier-to-distribution distances",
)

validate_distance_matrix(
    distance_dist_to_customer,
    expected_origins=len(distribution),
    expected_destinations=len(customers),
    matrix_name="Distribution-to-customer distances",
)


Supplier-to-distribution distances
----------------------------------
Shape: (16, 57)
Minimum distance: 0.0000 km
Maximum distance: 2,951.6968 km
Mean distance: 1,031.2780 km
Missing values: 0
Negative values: 0

Distribution-to-customer distances
----------------------------------
Shape: (57, 931)
Minimum distance: 0.0000 km
Maximum distance: 3,037.4060 km
Mean distance: 850.5212 km
Missing values: 0
Negative values: 0


In [22]:
nearest_dist_for_supplier = (
    distance_supplier_to_dist.idxmin(axis=1)
    .rename("nearest_dist_id")
    .to_frame()
)

nearest_dist_for_supplier["distance_km"] = (
    distance_supplier_to_dist.min(axis=1)
)

nearest_dist_for_supplier = (
    nearest_dist_for_supplier
    .reset_index()
    .rename(columns={"node_id": "supplier_id"})
)

nearest_dist_for_supplier

,supplier_id,nearest_dist_id,distance_km
0,R_KENAI,ANLTK02042,0.1288
1,R_PETROSTAR_NORTH_POLE,ANLTK02050,0.1519
2,R_PETROSTAR_VALDEZ,ANLTK02070,7.7503
3,R_CONOCOPHILLIPS_KUPARUK,D_KUPARUK_LOCAL,0.0000
4,R_HILCORP_PRUDHOE,ANLTK02054,0.1259
5,P_POA_ANCHORAGE,ANLTK02007,0.5115
6,P_HOMER,ANLTK02025,0.0769
7,P_NIKISKI,ANLTK02042,1.5603
8,P_UNALASKA,ANLTK02017,0.1283
9,P_NOME,ANLTK02048,0.6903


In [23]:
nearest_customer_for_dist = (
    distance_dist_to_customer.idxmin(axis=1)
    .rename("nearest_customer_id")
    .to_frame()
)

nearest_customer_for_dist["distance_km"] = (
    distance_dist_to_customer.min(axis=1)
)

nearest_customer_for_dist = (
    nearest_customer_for_dist
    .reset_index()
    .rename(columns={"node_id": "dist_id"})
)

nearest_customer_for_dist.head(20)

,dist_id,nearest_customer_id,distance_km
0,ANLTK02002,L_0081,0.6341
1,ANLTK02003,L_0446,1.7142
2,ANLTK02004,L_0081,0.7268
3,ANLTK02005,L_0081,1.0280
4,ANLTK02006,L_0081,0.8532
5,ANLTK02007,L_0112,0.6403
6,ANLTK02008,L_0444,148.7496
7,ANLTK02009,L_0051,0.4758
8,ANLTK02011,L_0675,1.2006
9,ANLTK02012,L_0665,1.3874


In [24]:
cost_supplier_to_dist = distance_supplier_to_dist.copy()
cost_dist_to_customer = distance_dist_to_customer.copy()

DISTANCE_COST_FACTOR = 1.0

cost_supplier_to_dist *= DISTANCE_COST_FACTOR
cost_dist_to_customer *= DISTANCE_COST_FACTOR

In [25]:
suppliers_model = (
    suppliers
    .copy()
    .set_index("node_id", drop=False)
)

distribution_model = (
    distribution
    .copy()
    .set_index("node_id", drop=False)
)

customers_model = (
    customers
    .copy()
    .set_index("node_id", drop=False)
)

In [26]:
S = suppliers_model.index.tolist()
D = distribution_model.index.tolist()
L = customers_model.index.tolist()

DUMMY = "DUMMY_UNUSED_SUPPLY"

print("Supplier nodes:", len(S))
print("Distribution nodes:", len(D))
print("Customer nodes:", len(L))

print("Supplier-to-D arcs:", len(S) * len(D))
print("D-to-customer arcs:", len(D) * len(L))

Supplier nodes: 16
Distribution nodes: 57
Customer nodes: 931
Supplier-to-D arcs: 912
D-to-customer arcs: 53067


In [27]:
supply = (
    suppliers_model["model_capacity"]
    .astype(float)
    .to_dict()
)

dist_capacity = (
    distribution_model["model_capacity"]
    .astype(float)
    .to_dict()
)

customer_demand = (
    customers_model["demand"]
    .astype(float)
    .to_dict()
)

In [28]:
total_supply = sum(supply.values())
total_demand = sum(customer_demand.values())

RHO_S = 0.10
target_total_supply = total_demand * (1 + RHO_S)

# assert np.isclose(
#     total_supply,
#     target_total_supply,
#     rtol=1e-10,
#     atol=1e-4,
# ), (
#     f"Supplier capacities must total 110% of demand. "
#     f"Expected {target_total_supply:,.4f}; found {total_supply:,.4f}."
# )

dummy_demand = total_supply - total_demand

print(f"Total supply:  {total_supply:,.4f}")
print(f"Total demand:  {total_demand:,.4f}")
print(f"Supply/demand ratio: {total_supply / total_demand:.4f}")
print(f"Dummy demand:  {dummy_demand:,.4f}")

assert dummy_demand >= -1e-6

# Avoid a tiny negative number caused by floating-point precision
dummy_demand = max(0.0, dummy_demand)

Total supply:  430,847.0809
Total demand:  391,679.1644
Supply/demand ratio: 1.1000
Dummy demand:  39,167.9164


In [29]:
cost_SD = {
    (s, d): float(cost_supplier_to_dist.loc[s, d])
    for s in S
    for d in D
}

cost_DL = {
    (d, l): float(cost_dist_to_customer.loc[d, l])
    for d in D
    for l in L
}

In [30]:
assert set(S) == set(supply)
assert set(D) == set(dist_capacity)
assert set(L) == set(customer_demand)

assert len(cost_SD) == len(S) * len(D)
assert len(cost_DL) == len(D) * len(L)

assert all(value >= 0 for value in supply.values())
assert all(value >= 0 for value in dist_capacity.values())
assert all(value >= 0 for value in customer_demand.values())
assert all(value >= 0 for value in cost_SD.values())
assert all(value >= 0 for value in cost_DL.values())

In [31]:
KUPARUK_R = "R_CONOCOPHILLIPS_KUPARUK"
PRUDHOE_R = "R_HILCORP_PRUDHOE"
VALDEZ_R = "R_PETROSTAR_VALDEZ"
VALDEZ_ALEUTIAN_TERMINALS = {
    "ANLTK02066",  # Westward Seafoods terminal
    "ANLTK02067",  # Captain's Bay Tank Farm
    "ANLTK02068",  # Resoff Tank Farm
}

required_suppliers = {KUPARUK_R, PRUDHOE_R, VALDEZ_R}
required_distribution_nodes = {
    KUPARUK_D, PRUDHOE_D, ADAK_D, ST_PAUL_D, ST_GEORGE_D,
    *VALDEZ_ALEUTIAN_TERMINALS,
}
required_customers = {
    KUPARUK_L, PRUDHOE_L, ADAK_L, ST_PAUL_L, ST_GEORGE_L
}

assert required_suppliers <= set(S)
assert required_distribution_nodes <= set(D)
assert required_customers <= set(L)

north_slope_paths = [
    (KUPARUK_R, KUPARUK_D, KUPARUK_L),
    (PRUDHOE_R, PRUDHOE_D, PRUDHOE_L),
]

island_terminal_paths = [
    (ADAK_D, ADAK_L),
    (ST_PAUL_D, ST_PAUL_L),
    (ST_GEORGE_D, ST_GEORGE_L),
]

for s, d, l in north_slope_paths:
    assert supply[s] + 1e-6 >= customer_demand[l], (
        f"Local supplier {s} cannot cover {l}."
    )
    assert dist_capacity[d] + 1e-6 >= customer_demand[l], (
        f"Local D node {d} cannot cover {l}."
    )

for d, l in island_terminal_paths:
    assert dist_capacity[d] + 1e-6 >= customer_demand[l], (
        f"Island D node {d} cannot cover {l}."
    )

forbidden_SD = set()
forbidden_DL = set()

# Northwest Alaska is supplied through marine distribution systems.
# The current public evidence does not support direct North Pole refinery
# deliveries to the Kotzebue or Red Dog marine terminals. These two arcs
# are therefore removed from the baseline network.
# Sources:
# https://dot.alaska.gov/nreg/nwatp/files/nwatpMarineRiverineConditions.pdf
# https://petrostar.com/divisions/refining/
north_pole_western_forbidden_SD = {
    ("R_PETROSTAR_NORTH_POLE", "ANLTK02035"),  # Kotzebue
    ("R_PETROSTAR_NORTH_POLE", "ANLTK02036"),  # Red Dog
}

#Red Dog (ANLTK02036) is now excluded
# from the D set as a mine port, which is why it can legitimately be absent. An
# unknown supplier id, by contrast, is a typo and still fails loudly.
unknown_restricted_suppliers = {
    s for s, _ in north_pole_western_forbidden_SD if s not in S
}
assert not unknown_restricted_suppliers, (
    f"Restricted arcs reference unknown suppliers: "
    f"{sorted(unknown_restricted_suppliers)}"
)

inapplicable_restricted_arcs = {
    (s, d) for s, d in north_pole_western_forbidden_SD if d not in D
}
if inapplicable_restricted_arcs:
    print(
        "Dropping restrictions on D nodes absent from the network:",
        sorted(inapplicable_restricted_arcs),
    )
north_pole_western_forbidden_SD -= inapplicable_restricted_arcs

forbidden_SD.update(north_pole_western_forbidden_SD)

for local_supplier, local_d, _ in north_slope_paths:
    forbidden_SD.update(
        (local_supplier, d) for d in D if d != local_d
    )
    forbidden_SD.update(
        (s, local_d) for s in S if s != local_supplier
    )

restricted_demand_paths = (
    [(d, l) for _, d, l in north_slope_paths]
    + island_terminal_paths
)
for local_d, local_customer in restricted_demand_paths:
    forbidden_DL.update(
        (local_d, l) for l in L if l != local_customer
    )
    forbidden_DL.update(
        (d, local_customer) for d in D if d != local_d
    )

# Hard regional service limits: a radius-limited terminal may not serve any
# customer beyond its service area. SERVICE_RADIUS_KM and the matching capacity
# pinning are defined in the capacity cell above; the distance matrix here is
# the same geodesic matrix used for the arc costs.
radius_forbidden_DL = {
    (d, l)
    for d, radius_km in SERVICE_RADIUS_KM.items()
    for l in L
    if float(distance_dist_to_customer.loc[d, l]) > radius_km
}
forbidden_DL.update(radius_forbidden_DL)
print("Radius-restricted D-to-customer arcs:", len(radius_forbidden_DL))

print("Forbidden supplier-to-D arcs:", len(forbidden_SD))
print("Forbidden D-to-customer arcs:", len(forbidden_DL))

Dropping restrictions on D nodes absent from the network: [('R_PETROSTAR_NORTH_POLE', 'ANLTK02036')]
Radius-restricted D-to-customer arcs: 5334
Forbidden supplier-to-D arcs: 141
Forbidden D-to-customer arcs: 10214


In [32]:
model = pulp.LpProblem(
    name="Alaska_Fuel_Distribution_Min_Cost_Flow",
    sense=pulp.LpMinimize,
)
flow_SD = pulp.LpVariable.dicts(
    name="Flow_SD",
    indexs=(S, D),
    lowBound=0,
    cat=pulp.LpContinuous,
)
flow_DL = pulp.LpVariable.dicts(
    name="Flow_DL",
    indexs=(D, L),
    lowBound=0,
    cat=pulp.LpContinuous,
)
unused_supply = pulp.LpVariable.dicts(
    name="Unused_Supply",
    indexs=S,
    lowBound=0,
    cat=pulp.LpContinuous,
)
supplier_to_dist_cost = pulp.lpSum(
    cost_SD[s, d] * flow_SD[s][d]
    for s in S
    for d in D
)

dist_to_customer_cost = pulp.lpSum(
    cost_DL[d, l] * flow_DL[d][l]
    for d in D
    for l in L
)

model += (
    supplier_to_dist_cost + dist_to_customer_cost,
    "Total_Transportation_Cost",
)
for s in S:
    model += (
        pulp.lpSum(flow_SD[s][d] for d in D)
        + unused_supply[s]
        == supply[s],
        f"Supplier_Balance_{s}",
    )

model += (
    pulp.lpSum(unused_supply[s] for s in S)
    == dummy_demand,
    "Dummy_Demand_Satisfaction",
)
for d in D:
    model += (
        pulp.lpSum(flow_SD[s][d] for s in S)
        ==
        pulp.lpSum(flow_DL[d][l] for l in L),
        f"Distribution_Flow_Balance_{d}",
    )
for d in D:
    model += (
        pulp.lpSum(flow_DL[d][l] for l in L)
        <= dist_capacity[d],
        f"Distribution_Capacity_{d}",
    )
for l in L:
    model += (
        pulp.lpSum(flow_DL[d][l] for d in D)
        == customer_demand[l],
        f"Customer_Demand_{l}",
    )

for restriction_number, (s, d) in enumerate(sorted(forbidden_SD)):
    model += (
        flow_SD[s][d] == 0,
        f"Forbidden_SD_{restriction_number}",
    )

for restriction_number, (d, l) in enumerate(sorted(forbidden_DL)):
    model += (
        flow_DL[d][l] == 0,
        f"Forbidden_DL_{restriction_number}",
    )

# Petro Star confirms refined-product barge movements between its Valdez
# system and the Aleutian Islands. This is a one-barrel activation
# constraint: it fixes the business topology without imposing an
# unsupported shipment share in this aggregate, single-product model.
# Source: https://petrostar.com/locations/
MIN_VALDEZ_ALEUTIAN_FLOW = 1.0
model += (
    pulp.lpSum(
        flow_SD[VALDEZ_R][d]
        for d in VALDEZ_ALEUTIAN_TERMINALS
    )
    >= MIN_VALDEZ_ALEUTIAN_FLOW,
    "Minimum_Valdez_to_Aleutian_Terminal_Group",
)

# MIN_ACTIVE_TERMINAL_FLOW = 1.0

# for d in confirmed_active_terminal_ids:
#     model += (
#         pulp.lpSum(flow_DL[d][l] for l in L)
#         >= MIN_ACTIVE_TERMINAL_FLOW,
#         f"Minimum_Active_Throughput_{d}",
#     )


In [33]:
print("Model name:", model.name)
print("Number of variables:", len(model.variables()))
print("Number of constraints:", len(model.constraints))

expected_variables = (
    len(S) * len(D)
    + len(D) * len(L)
    + len(S)
)

base_expected_constraints = (
    len(S)       # Supplier balances
    + 1          # Dummy-demand balance
    + len(D)     # Terminal flow balances
    + len(D)     # Terminal capacities
    + len(L)     # Customer demands
)
restriction_constraints = len(forbidden_SD) + len(forbidden_DL)
business_connectivity_constraints = 1
expected_constraints = (
    base_expected_constraints
    + restriction_constraints
    + business_connectivity_constraints
)

print("Expected variables:", expected_variables)
print("Base constraints:", base_expected_constraints)
print("Arc-restriction constraints:", restriction_constraints)
print("Business-connectivity constraints:", business_connectivity_constraints)
print("Expected constraints:", expected_constraints)

assert len(model.variables()) == expected_variables
assert len(model.constraints) == expected_constraints

Model name: Alaska_Fuel_Distribution_Min_Cost_Flow
Number of variables: 53995
Number of constraints: 11418
Expected variables: 53995
Base constraints: 1062
Arc-restriction constraints: 10355
Business-connectivity constraints: 1
Expected constraints: 11418


In [34]:
# The reference network was constructed with Gurobi 12.0.3.
solver = pulp.GUROBI_CMD(msg=True)
# If gurobi_cl is not on PATH, provide its local executable path:
# solver = pulp.GUROBI_CMD(path="/path/to/gurobi_cl", msg=True)
if not solver.available():
    raise RuntimeError("Gurobi is unavailable. Install and license Gurobi 12.0.3 and place gurobi_cl on PATH.")

status_code = model.solve(solver)

solver_status = pulp.LpStatus[status_code]
objective_value = float(
    pulp.value(model.objective)
)

print("Solver status:", solver_status)
print(
    "Objective value:",
    f"{objective_value:,.4f} barrel-km",
)

if solver_status != "Optimal":
    raise RuntimeError(
        f"Optimization ended with status: {solver_status}"
    )

Set parameter Username
Set parameter LicenseID to value 2719156
Set parameter LogFile to value "gurobi.log"
Using license file /Users/alicanyilmaz/gurobi.lic
Academic license - for non-commercial use only - expires 2026-10-07

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[rosetta2] - Darwin 23.6.0 23G93)
Copyright (c) 2025, Gurobi Optimization, LLC

Read LP format model from file /var/folders/td/k8ljb3y104lfc427_10jdv6h0000gn/T/0e468f4d781543d18bc1de6c9a5e6768-pulp.lp
Reading time = 0.07 seconds
Total_Transportation_Cost: 11418 rows, 53995 columns, 171415 nonzeros

Using Gurobi shared library /Library/gurobi1203/macos_universal2/lib/libgurobi120.dylib

CPU model: Apple M3 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 11418 rows, 53995 columns and 171415 nonzeros
Model fingerprint: 0xd1cda5b6
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [6e-02, 3e+03]
  Bounds range     [0e+00, 0e+00]
 

In [35]:
FLOW_TOLERANCE = 1e-5
north_slope_rows = []

for s, d, l in north_slope_paths:
    supplier_to_d_flow = float(pulp.value(flow_SD[s][d]) or 0.0)
    d_to_customer_flow = float(pulp.value(flow_DL[d][l]) or 0.0)
    required_flow = float(customer_demand[l])

    assert abs(supplier_to_d_flow - required_flow) <= FLOW_TOLERANCE
    assert abs(d_to_customer_flow - required_flow) <= FLOW_TOLERANCE

    north_slope_rows.append(
        {
            "supplier_id": s,
            "distribution_id": d,
            "customer_id": l,
            "required_demand": required_flow,
            "supplier_to_D_flow": supplier_to_d_flow,
            "D_to_customer_flow": d_to_customer_flow,
        }
    )

island_rows = []
for d, l in island_terminal_paths:
    d_to_customer_flow = float(pulp.value(flow_DL[d][l]) or 0.0)
    supplier_inflow = sum(
        float(pulp.value(flow_SD[s][d]) or 0.0) for s in S
    )
    required_flow = float(customer_demand[l])

    assert abs(supplier_inflow - required_flow) <= FLOW_TOLERANCE
    assert abs(d_to_customer_flow - required_flow) <= FLOW_TOLERANCE

    island_rows.append(
        {
            "distribution_id": d,
            "customer_id": l,
            "required_demand": required_flow,
            "supplier_to_D_flow": supplier_inflow,
            "D_to_customer_flow": d_to_customer_flow,
        }
    )

max_forbidden_SD_flow = max(
    (float(pulp.value(flow_SD[s][d]) or 0.0) for s, d in forbidden_SD),
    default=0.0,
)
max_forbidden_DL_flow = max(
    (float(pulp.value(flow_DL[d][l]) or 0.0) for d, l in forbidden_DL),
    default=0.0,
)

assert max_forbidden_SD_flow <= FLOW_TOLERANCE
assert max_forbidden_DL_flow <= FLOW_TOLERANCE

valdez_aleutian_flow = sum(
    float(pulp.value(flow_SD[VALDEZ_R][d]) or 0.0)
    for d in VALDEZ_ALEUTIAN_TERMINALS
)
assert (
    valdez_aleutian_flow + FLOW_TOLERANCE
    >= MIN_VALDEZ_ALEUTIAN_FLOW
)
print(
    "Valdez refinery to Aleutian terminal-group flow:",
    f"{valdez_aleutian_flow:,.4f}",
)

north_slope_flow_check = pd.DataFrame(north_slope_rows)
island_flow_check = pd.DataFrame(island_rows)
display(north_slope_flow_check)
display(island_flow_check)

Valdez refinery to Aleutian terminal-group flow: 1.0000


,supplier_id,distribution_id,customer_id,required_demand,supplier_to_D_flow,D_to_customer_flow
0,R_CONOCOPHILLIPS_KUPARUK,D_KUPARUK_LOCAL,L_KUPARUK_INDUSTRIAL_OPERATIONS,"20,131.0930","20,131.0930","20,131.0930"
1,R_HILCORP_PRUDHOE,ANLTK02054,L_PRUDHOE_DEADHORSE_INDUSTRIAL_OPERATIONS,"8,723.4740","8,723.4740","8,723.4740"


,distribution_id,customer_id,required_demand,supplier_to_D_flow,D_to_customer_flow
0,ANLTK02001,L_0039,"1,692.8571","1,692.8571","1,692.8571"
1,ANLTK02063,L_ST_PAUL_COMMUNITY_FUEL,"4,088.5965","4,088.5965","4,088.5965"
2,ANLTK02062,L_ST_GEORGE_COMMUNITY_FUEL,663.2832,663.2832,663.2832


In [36]:
def make_geodesic_linestring_wkt(
    origin_lon,
    origin_lat,
    destination_lon,
    destination_lat,
    max_segment_km=50,
):
    """
    Return a densified WGS84 geodesic line as WKT.

    Coordinate order in WKT is longitude latitude.
    """

    _, _, distance_m = geod.inv(
        origin_lon,
        origin_lat,
        destination_lon,
        destination_lat,
    )

    distance_km = distance_m / 1_000

    number_of_intermediate_points = max(
        0,
        int(np.ceil(distance_km / max_segment_km)) - 1,
    )

    coordinates = [
        (origin_lon, origin_lat)
    ]

    if number_of_intermediate_points > 0:
        intermediate_points = geod.npts(
            origin_lon,
            origin_lat,
            destination_lon,
            destination_lat,
            number_of_intermediate_points,
        )

        coordinates.extend(intermediate_points)

    coordinates.append(
        (destination_lon, destination_lat)
    )

    coordinate_text = ", ".join(
        f"{lon:.8f} {lat:.8f}"
        for lon, lat in coordinates
    )

    return f"LINESTRING ({coordinate_text})"

In [37]:
test_supplier = suppliers_model.iloc[0]
test_terminal = distribution_model.iloc[0]

test_wkt = make_geodesic_linestring_wkt(
    origin_lon=test_supplier["longitude"],
    origin_lat=test_supplier["latitude"],
    destination_lon=test_terminal["longitude"],
    destination_lat=test_terminal["latitude"],
)

print(test_wkt[:300])

LINESTRING (-151.36722200 60.68416700, -150.87883176 60.86935286, -150.38478047 61.05273996, -149.88501600 61.23429400)


In [38]:
FLOW_TOLERANCE = 1e-6
supplier_to_dist_records = []

for s in S:
    supplier = suppliers_model.loc[s]

    for d in D:
        flow_value = pulp.value(flow_SD[s][d])

        if flow_value is None or flow_value <= FLOW_TOLERANCE:
            continue

        terminal = distribution_model.loc[d]
        distance_km = cost_SD[s, d]

        supplier_type = supplier["node_type"]

        supplier_to_dist_records.append(
            {
                "arc_id": f"{supplier_type}D_{s}_{d}",
                "arc_stage": f"{supplier_type}-D",

                "origin_id": s,
                "origin_name": supplier["node_name"],
                "origin_type": supplier_type,
                "origin_latitude": supplier["latitude"],
                "origin_longitude": supplier["longitude"],

                "destination_id": d,
                "destination_name": terminal["node_name"],
                "destination_type": "D",
                "destination_latitude": terminal["latitude"],
                "destination_longitude": terminal["longitude"],

                "flow_barrels": flow_value,
                "distance_km": distance_km,
                "unit_cost": distance_km,
                "arc_cost_barrel_km": flow_value * distance_km,

                "geometry_wkt": make_geodesic_linestring_wkt(
                    origin_lon=supplier["longitude"],
                    origin_lat=supplier["latitude"],
                    destination_lon=terminal["longitude"],
                    destination_lat=terminal["latitude"],
                ),
            }
        )

arcs_SD = pd.DataFrame(supplier_to_dist_records)

print("Positive supplier-to-terminal arcs:", len(arcs_SD))
arcs_SD.head()

Positive supplier-to-terminal arcs: 61


,arc_id,arc_stage,origin_id,origin_name,origin_type,origin_latitude,origin_longitude,destination_id,destination_name,destination_type,destination_latitude,destination_longitude,flow_barrels,distance_km,unit_cost,arc_cost_barrel_km,geometry_wkt
0,RD_R_KENAI_ANLTK02003,R-D,R_KENAI,Kenai Refinery,R,60.6842,-151.3672,ANLTK02003,ANCHORAGE FUELING AND SERVICE COMPANY AIRPORT ...,D,61.1749,-150.0113,"61,774.7861",91.6335,91.6335,"5,660,639.4800","LINESTRING (-151.36722200 60.68416700, -150.69..."
1,RD_R_KENAI_ANLTK02004,R-D,R_KENAI,Kenai Refinery,R,60.6842,-151.3672,ANLTK02004,CROWLEY FUELS LLC - ANCHORAGE BULK FUEL FACILITY,D,61.2312,-149.8882,"8,565.9384",100.6748,100.6748,"862,374.5370","LINESTRING (-151.36722200 60.68416700, -150.87..."
2,RD_R_KENAI_ANLTK02005,R-D,R_KENAI,Kenai Refinery,R,60.6842,-151.3672,ANLTK02005,TESORO LOGISTICS OPERATIONS LLC OCEAN DOCK PET...,D,61.2302,-149.8937,"38,637.0859",100.3788,100.3788,"3,878,345.9750","LINESTRING (-151.36722200 60.68416700, -150.88..."
3,RD_R_KENAI_ANLTK02042,R-D,R_KENAI,Kenai Refinery,R,60.6842,-151.3672,ANLTK02042,ANDEAVOR LOGISTICS LP - KENAI REFINERY STORAGE...,D,60.6834,-151.3654,"14,935.9069",0.1288,0.1288,"1,923.8384","LINESTRING (-151.36722200 60.68416700, -151.36..."
4,RD_R_PETROSTAR_NORTH_POLE_ANLTK02019,R-D,R_PETROSTAR_NORTH_POLE,Petro Star North Pole Refinery,R,64.7327,-147.3453,ANLTK02019,US DEPARTMENT OF THE AIR FORCE - EIELSON AIR F...,D,64.6584,-147.0508,"6,316.9612",16.3088,16.3088,"103,022.2563","LINESTRING (-147.34530400 64.73266700, -147.05..."


In [39]:
dist_to_customer_records = []

for d in D:
    terminal = distribution_model.loc[d]

    for l in L:
        flow_value = pulp.value(flow_DL[d][l])

        if flow_value is None or flow_value <= FLOW_TOLERANCE:
            continue

        customer = customers_model.loc[l]
        distance_km = cost_DL[d, l]

        dist_to_customer_records.append(
            {
                "arc_id": f"DL_{d}_{l}",
                "arc_stage": "D-L",

                "origin_id": d,
                "origin_name": terminal["node_name"],
                "origin_type": "D",
                "origin_latitude": terminal["latitude"],
                "origin_longitude": terminal["longitude"],

                "destination_id": l,
                "destination_name": customer["node_name"],
                "destination_type": "L",
                "destination_latitude": customer["latitude"],
                "destination_longitude": customer["longitude"],

                "flow_barrels": flow_value,
                "distance_km": distance_km,
                "unit_cost": distance_km,
                "arc_cost_barrel_km": flow_value * distance_km,

                "geometry_wkt": make_geodesic_linestring_wkt(
                    origin_lon=terminal["longitude"],
                    origin_lat=terminal["latitude"],
                    destination_lon=customer["longitude"],
                    destination_lat=customer["latitude"],
                ),
            }
        )

arcs_DL = pd.DataFrame(dist_to_customer_records)

print("Positive terminal-to-customer arcs:", len(arcs_DL))
arcs_DL.head()

Positive terminal-to-customer arcs: 961


,arc_id,arc_stage,origin_id,origin_name,origin_type,origin_latitude,origin_longitude,destination_id,destination_name,destination_type,destination_latitude,destination_longitude,flow_barrels,distance_km,unit_cost,arc_cost_barrel_km,geometry_wkt
0,DL_ANLTK02002_L_0001,D-L,ANLTK02002,ANCHORAGE FUELING AND SERVICE COMPANY OFF-AIRP...,D,61.2343,-149.8850,L_0001,"""Alice Mae's"" Shoppers Cache - The Laundry",L,61.3941,-149.4634,238.0952,28.7654,28.7654,"6,848.9087","LINESTRING (-149.88501600 61.23429400, -149.46..."
1,DL_ANLTK02002_L_0004,D-L,ANLTK02002,ANCHORAGE FUELING AND SERVICE COMPANY OFF-AIRP...,D,61.2343,-149.8850,L_0004,2 Go Mart-Tesoro Northstore #0077,L,61.5938,-149.1211,380.9524,57.1733,57.1733,"21,780.2914","LINESTRING (-149.88501600 61.23429400, -149.50..."
2,DL_ANLTK02002_L_0026,D-L,ANLTK02002,ANCHORAGE FUELING AND SERVICE COMPANY OFF-AIRP...,D,61.2343,-149.8850,L_0026,2Go Mart-Tesoro Northstore #73,L,61.3298,-149.5666,476.1905,20.1188,20.1188,"9,580.3956","LINESTRING (-149.88501600 61.23429400, -149.56..."
3,DL_ANLTK02002_L_0031,D-L,ANLTK02002,ANCHORAGE FUELING AND SERVICE COMPANY OFF-AIRP...,D,61.2343,-149.8850,L_0031,673 CES/CEIEC,L,61.2493,-148.1963,208.3333,90.6882,90.6882,"18,893.3683","LINESTRING (-149.88501600 61.23429400, -149.04..."
4,DL_ANLTK02002_L_0035,D-L,ANLTK02002,ANCHORAGE FUELING AND SERVICE COMPANY OFF-AIRP...,D,61.2343,-149.8850,L_0035,AAFES Gas Station,L,61.2371,-149.8466,385.0281,2.0882,2.0882,804.0258,"LINESTRING (-149.88501600 61.23429400, -149.84..."


In [40]:
all_arcs = pd.concat(
    [arcs_SD, arcs_DL],
    ignore_index=True,
)

all_arcs = all_arcs.sort_values(
    by=["arc_stage", "origin_id", "destination_id"]
).reset_index(drop=True)

print("Total positive physical arcs:", len(all_arcs))

all_arcs.groupby("arc_stage").agg(
    number_of_arcs=("arc_id", "count"),
    total_flow_barrels=("flow_barrels", "sum"),
    total_cost_barrel_km=("arc_cost_barrel_km", "sum"),
)

Total positive physical arcs: 1022


,number_of_arcs,total_flow_barrels,total_cost_barrel_km
arc_stage,,,
D-L,961,"391,679.1644","38,738,544.0026"
P-D,33,"103,633.5444","6,771,388.8014"
R-D,28,"288,045.6200","36,216,826.3643"


In [41]:
flow_entering_distribution = arcs_SD["flow_barrels"].sum()
flow_leaving_distribution = arcs_DL["flow_barrels"].sum()

extracted_objective = (
    arcs_SD["arc_cost_barrel_km"].sum()
    + arcs_DL["arc_cost_barrel_km"].sum()
)

print(
    "Flow entering distribution:",
    f"{flow_entering_distribution:,.4f}",
)

print(
    "Flow leaving distribution:",
    f"{flow_leaving_distribution:,.4f}",
)

print(
    "Customer demand:",
    f"{total_demand:,.4f}",
)

print(
    "Model objective:",
    f"{objective_value:,.4f}",
)

print(
    "Objective calculated from arcs:",
    f"{extracted_objective:,.4f}",
)

Flow entering distribution: 391,679.1644
Flow leaving distribution: 391,679.1644
Customer demand: 391,679.1644
Model objective: 81,726,759.1683
Objective calculated from arcs: 81,726,759.1683


In [42]:
assert np.isclose(
    flow_entering_distribution,
    total_demand,
    rtol=1e-7,
    atol=1e-4,
)

assert np.isclose(
    flow_leaving_distribution,
    total_demand,
    rtol=1e-7,
    atol=1e-4,
)

assert np.isclose(
    extracted_objective,
    objective_value,
    rtol=1e-7,
    atol=1e-2,
)

In [43]:
# Write directly into data/processed/ 
RESULTS_DIR = DATA_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SD_ARCS_FILE = (
    RESULTS_DIR
    / "supplier_to_terminal_arcs.csv"
)

DL_ARCS_FILE = (
    RESULTS_DIR
    / "terminal_to_last_mile_arcs.csv"
)

ALL_ARCS_FILE = (
    RESULTS_DIR
    / "network_arcs.csv"
)

arcs_SD.to_csv(
    SD_ARCS_FILE,
    index=False,
    float_format="%.8f",
)

arcs_DL.to_csv(
    DL_ARCS_FILE,
    index=False,
    float_format="%.8f",
)

all_arcs.to_csv(
    ALL_ARCS_FILE,
    index=False,
    float_format="%.8f",
)

print("Files created:")
print(SD_ARCS_FILE)
print(DL_ARCS_FILE)
print(ALL_ARCS_FILE)

Files created:
/Users/alicanyilmaz/Desktop/alaska-fuel-network/data/processed/supplier_to_terminal_arcs.csv
/Users/alicanyilmaz/Desktop/alaska-fuel-network/data/processed/terminal_to_last_mile_arcs.csv
/Users/alicanyilmaz/Desktop/alaska-fuel-network/data/processed/network_arcs.csv
